<a href="https://colab.research.google.com/github/hcoo0019-sys/MMA3001-Project/blob/main/MMA3001_Project_Demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Load Model Artifacts from Extracted Zip

Please download your model artifacts ZIP file and extract its contents directly into your main Colab working directory (so that `model.pt`, `scaler.pkl`, and `config.json` reside directly inside `/content/`).

[5]
0s


In [ ]:
import json
import joblib
import torch
import os

# Look directly in the main Colab directory
ARTIFACTS_DIR = "/content"

required_files = ["model.pt", "scaler.pkl", "config.json"]
missing_files = [f for f in required_files if not os.path.exists(os.path.join(ARTIFACTS_DIR, f))]

if missing_files:
    raise FileNotFoundError(
        f"Missing required artifact files in '{ARTIFACTS_DIR}/': {missing_files}. "
        f"Please make sure you have extracted the ZIP file directly into the main '/content/' directory."
    )

# Load the configuration and fitted scaler
with open(os.path.join(ARTIFACTS_DIR, "config.json"), "r") as f:
    config = json.load(f)

scaler = joblib.load(os.path.join(ARTIFACTS_DIR, "scaler.pkl"))

print("\nAll local model artifacts verified and loaded successfully.")
print("Features:", len(config["feature_columns"]))
print("Maximum RUL:", config["max_rul"])


All local model artifacts verified and loaded successfully.
Features: 15
Maximum RUL: 361.0


# Load Neural Network

In [ ]:
import torch
import torch.nn as nn
import os

# Recreate the exact architecture used during training
class UpgradedRULPredictor(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.Mish(),
            nn.Dropout(0.15),
            nn.Linear(128, 64),
            nn.Mish(),
            nn.Dropout(0.15),
            nn.Linear(64, 32),
            nn.Mish(),
            nn.Linear(32, 1)
        )

    def forward(self, x):
        return self.net(x)

# Initialise model and load trained weights
model = UpgradedRULPredictor(
    input_dim=len(config["feature_columns"])
)

state_dict = torch.load(
    os.path.join(ARTIFACTS_DIR, "model.pt"),
    map_location="cpu",
    weights_only=True
)

model.load_state_dict(state_dict)
model.eval()

print("Neural network loaded and ready for predictions.")

Neural network loaded and ready for predictions.


# Upload Data

In [ ]:


import pandas as pd
import numpy as np
from google.colab import files
import io

columns = [
    "engine_id", "cycle",
    "setting_1", "setting_2", "setting_3",
    "sensor_1", "sensor_2", "sensor_3", "sensor_4",
    "sensor_5", "sensor_6", "sensor_7", "sensor_8",
    "sensor_9", "sensor_10", "sensor_11", "sensor_12",
    "sensor_13", "sensor_14", "sensor_15", "sensor_16",
    "sensor_17", "sensor_18", "sensor_19", "sensor_20",
    "sensor_21"
]

answer = input("Does your file have column headers? (y/n): ").strip().lower()

if answer not in ["y", "yes", "n", "no"]:
    raise ValueError("Please enter y or n.")

has_headers = answer in ["y", "yes"]

print("Upload engine data (.txt or .csv):")
uploaded = files.upload()

if not uploaded:
    raise ValueError("No file was uploaded.")

filename = next(iter(uploaded))
file_bytes = io.BytesIO(uploaded[filename])

# Import the file, ignoring any existing column names
if filename.lower().endswith(".csv"):
    df = pd.read_csv(
        file_bytes,
        header=0 if has_headers else None
    )
else:
    df = pd.read_csv(
        file_bytes,
        sep=r"\s+",
        header=0 if has_headers else None
    )

# Check column count
if df.shape[1] != len(columns):
    raise ValueError(
        f"Expected 26 columns, but found {df.shape[1]}."
    )

# Replace all existing headings with our standard names
df.columns = columns

# Convert all columns to numeric
df = df.apply(pd.to_numeric, errors="coerce")

if df.empty:
    raise ValueError("The uploaded file contains no engine observations.")

if df.isna().any().any():
    raise ValueError("Missing or non-numeric values found in the uploaded file.")

for col in ["engine_id", "cycle"]:
    if (df[col] <= 0).any() or (df[col] % 1 != 0).any():
        raise ValueError(f"'{col}' must contain positive integers.")

if df.duplicated(["engine_id", "cycle"]).any():
    raise ValueError("Duplicate engine ID and cycle combinations found.")

df = df.sort_values(["engine_id", "cycle"]).reset_index(drop=True)

print(f"\nSuccessfully loaded {filename}")
print(f"Headers ignored: {has_headers}")
print(f"Engines: {df['engine_id'].nunique()}")
print(f"Observations: {len(df)}")
display(df.head())


Does your file have column headers? (y/n): y
Upload engine data (.txt or .csv):


Saving train_FD001.txt to train_FD001 (1).txt

Successfully loaded train_FD001 (1).txt
Headers ignored: True
Engines: 100
Observations: 20630


,engine_id,cycle,setting_1,setting_2,setting_3,sensor_1,sensor_2,sensor_3,sensor_4,sensor_5,...,sensor_12,sensor_13,sensor_14,sensor_15,sensor_16,sensor_17,sensor_18,sensor_19,sensor_20,sensor_21
0,1,2,0.0019,-0.0003,100.0,518.67,642.15,1591.82,1403.14,14.62,...,522.28,2388.07,8131.49,8.4318,0.03,392,2388,100.0,39.00,23.4236
1,1,3,-0.0043,0.0003,100.0,518.67,642.35,1587.99,1404.20,14.62,...,522.42,2388.03,8133.23,8.4178,0.03,390,2388,100.0,38.95,23.3442
2,1,4,0.0007,0.0000,100.0,518.67,642.35,1582.79,1401.87,14.62,...,522.86,2388.08,8133.83,8.3682,0.03,392,2388,100.0,38.88,23.3739
3,1,5,-0.0019,-0.0002,100.0,518.67,642.37,1582.85,1406.22,14.62,...,522.19,2388.04,8133.80,8.4294,0.03,393,2388,100.0,38.90,23.4044
4,1,6,-0.0043,-0.0001,100.0,518.67,642.10,1584.47,1398.37,14.62,...,521.68,2388.03,8132.85,8.4108,0.03,391,2388,100.0,38.98,23.3669


# Validate Input

In [ ]:

# ==========================================
# Validate uploaded engine data
# ==========================================

sensor_cols = [col for col in columns if col.startswith("sensor_")]
required_cols = ["engine_id", "cycle"] + sensor_cols

# 1. Check required columns exist
missing_cols = [col for col in required_cols if col not in df.columns]

if missing_cols:
    raise ValueError(f"Missing required columns: {missing_cols}")

# 2. Convert required values to numeric, then check for missing/invalid values
for col in required_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

missing_counts = df[required_cols].isna().sum()
missing_counts = missing_counts[missing_counts > 0]

if not missing_counts.empty:
    print("Missing or non-numeric values detected:")
    print(missing_counts)
    raise ValueError("Prediction stopped: correct the invalid values.")

# 3. Check engine IDs and cycles are positive integers
for col in ["engine_id", "cycle"]:
    if (df[col] <= 0).any() or (df[col] % 1 != 0).any():
        raise ValueError(f"'{col}' must contain positive integers.")

# 4. Check for duplicate engine-cycle observations
duplicates = df.duplicated(subset=["engine_id", "cycle"])

if duplicates.any():
    raise ValueError(
        f"{duplicates.sum()} duplicate engine-cycle rows found."
    )

# 5. Sort the observations
df = df.sort_values(["engine_id", "cycle"]).reset_index(drop=True)

print("Basic data validation passed.")
print(f"Engines: {df['engine_id'].nunique()}")
print(f"Observations: {len(df)}")


Basic data validation passed.
Engines: 100
Observations: 20630


In [ ]:

# ==========================================
# Validate sensor values against FD001 ranges
# ==========================================

SENSOR_RANGES = {
    "sensor_1": (518.67, 518.67),
    "sensor_2": (641.21, 644.53),
    "sensor_3": (1571.04, 1616.91),
    "sensor_4": (1382.25, 1441.49),
    "sensor_5": (14.62, 14.62),
    "sensor_6": (21.6, 21.61),
    "sensor_7": (549.85, 556.06),
    "sensor_8": (2387.9, 2388.56),
    "sensor_9": (9021.73, 9244.59),
    "sensor_10": (1.3, 1.3),
    "sensor_11": (46.85, 48.53),
    "sensor_12": (518.69, 523.38),
    "sensor_13": (2387.88, 2388.56),
    "sensor_14": (8099.94, 8293.72),
    "sensor_15": (8.3249, 8.5848),
    "sensor_16": (0.03, 0.03),
    "sensor_17": (388.0, 400.0),
    "sensor_18": (2388.0, 2388.0),
    "sensor_19": (100.0, 100.0),
    "sensor_20": (38.14, 39.43),
    "sensor_21": (22.8942, 23.6184)
}

# Check each sensor against its training-data range
range_issues = []

for sensor, (min_val, max_val) in SENSOR_RANGES.items():
    if sensor not in df.columns:
        raise ValueError(f"Required sensor column missing: {sensor}")

    outside = (df[sensor] < min_val) | (df[sensor] > max_val)

    if outside.any():
        range_issues.append({
            "Sensor": sensor,
            "Training minimum": min_val,
            "Training maximum": max_val,
            "Out-of-range observations": int(outside.sum())
        })

if range_issues:
    print("WARNING: Some sensor values are outside the FD001 training ranges.")
    print("Predictions may be less reliable for these observations.")
    display(pd.DataFrame(range_issues))
else:
    print("All sensor values are within the FD001 training ranges.")


All sensor values are within the FD001 training ranges.


# RUL Prediction

In [ ]:

# ==========================================
# Predict RUL for each engine
# ==========================================

# Get the latest recorded observation for each engine
latest_cycles = (
    df.sort_values(["engine_id", "cycle"])
      .groupby("engine_id")
      .tail(1)
      .sort_values("engine_id")
      .reset_index(drop=True)
)

# Use the exact features and order from model training
feature_cols = config["feature_columns"]

missing_features = [col for col in feature_cols if col not in latest_cycles.columns]
if missing_features:
    raise ValueError(f"Missing model input features: {missing_features}")

X_latest = latest_cycles[feature_cols].values

# Apply the fitted training scaler
X_latest_scaled = scaler.transform(X_latest)

# Convert to PyTorch tensor
X_tensor = torch.tensor(X_latest_scaled, dtype=torch.float32)

# Generate predictions
model.eval()
with torch.no_grad():
    predictions = model(X_tensor).squeeze(1).numpy()

# Apply the same prediction limits used during validation
predictions = np.clip(predictions, 0, config["max_rul"])

# Create results table
results = pd.DataFrame({
    "engine_id": latest_cycles["engine_id"].astype(int),
    "last_observed_cycle": latest_cycles["cycle"].astype(int),
    "predicted_RUL": np.round(predictions, 1)
})

# Flag engines whose predicted RUL is at or below the warning threshold
results["warning"] = np.where(
    results["predicted_RUL"] <= config["warning_rul_threshold"],
    "WARNING",
    "Normal"
)

print(f"Predictions generated for {len(results)} engines.")
display(results)


Predictions generated for 100 engines.


,engine_id,last_observed_cycle,predicted_RUL,warning
0,1,192,5.8,WARNING
1,2,287,6.4,WARNING
2,3,179,3.0,WARNING
3,4,189,3.3,WARNING
4,5,269,4.1,WARNING
...,...,...,...,...
95,96,336,4.2,WARNING
96,97,202,3.4,WARNING
97,98,156,5.7,WARNING
98,99,185,8.5,WARNING
